# Lab B: FAISS Retrieval, Local Answer Extraction, and Simple Evaluation

> **How long:** about 50 minutes  
> **Session shape:** the morning slot explains retrievers and FAISS; this notebook is the afternoon build-and-verify lab.  
> **Goal:** extend `smartLearn-AI/smartlearn-backend/services/rag.py` so the project can build a FAISS index, retrieve top-k chunks, answer from retrieved evidence, and measure a few short questions.  
> **Checkpoint:** `pdf1.pdf` and `pdf2.pdf` both return page-aware retrieval results, and `section2_eval_results.csv` is saved under `Day3/artifacts/reports/`.

> **CPU-first:** FAISS stays on the stable local CPU path. A GPU only changes embedding speed; it does not change the retrieval design.


## Before You Run

This notebook is the afternoon implementation lab. The morning lecture should already cover retriever design, similarity search, and why FAISS helps with longer PDFs.

You should first finish Lab A. Keep using the same environment.

**Windows / cmd**

```bash
cd smartLearn-AI
.\.venv\Scripts\activate.bat
```

**macOS / Linux**

```bash
cd smartLearn-AI
source .venv/bin/activate
```


> **How to use this notebook - Balanced Vibe Coding Mode**
>
> Stay disciplined about scope:
>
> - **Do directly:** run retrieval checks, inspect top-k chunks, compare hit pages, read `git diff`, and save the evaluation report.
> - **Use Claude Code:** add or revise FAISS helpers, retrieval wrappers, answer extraction, and the project-facing document shape inside `smartlearn-backend/services/rag.py`.
>
> Each coding step should answer one question: what new visible behavior should appear after this edit?
>
> If `rag.py` is brand-new and `git diff -- smartlearn-backend/services/rag.py` shows nothing, run `git add -N smartlearn-backend/services/rag.py` once and then check the diff again.
>
> The notebook verification cells are the evidence. If a retrieval cell still fails, do not move on to evaluation yet.


## 2.1 Define "Done" Before Editing

### Purpose

This lab is about retrieval and evidence flow.

Write these acceptance criteria before editing:

1. `rag.py` can build or load a FAISS index from the saved embeddings.
2. `rag.py` can retrieve top-k chunks with page numbers.
3. `rag.py` can return a simple answer from retrieved evidence even without an LLM API call.
4. `rag.py` can package its outputs in a shape that later fits the Day 2 `documents[chat_id]` store.
5. The notebook can measure a few short-answer questions on `pdf1.pdf` and `pdf2.pdf`.


<!-- BEGINNER-WALKTHROUGH -->

### Beginner walkthrough: name the two separate jobs

Before touching code, say these two sentences aloud:

1. `Embeddings + FAISS choose evidence.`
2. `Answer generation explains that evidence.`

That split matters because a system can retrieve the right chunk but still phrase a weak answer.

**Completion evidence:** you can explain why a retrieval hit rate and an answer hit rate are different numbers.


> **Checkpoint 1 - Retrieval and answering are separated:** you can say which part picks chunks and which part turns those chunks into a sentence.


## 2.2 Reuse the Lab A Outputs

### Purpose

Do not rebuild the project structure. Lab B should reuse the same `rag.py`, the same active config, and any Lab 1 artifacts that are already present. If a needed artifact is missing or stale, rebuild it.


In [1]:
import importlib
import sys
from pathlib import Path
from IPython.display import display

import pandas as pd

cwd = Path.cwd()
repo_name_candidates = ["smartLearn-AI"]
path_pairs = []
for repo_name in repo_name_candidates:
    path_pairs.extend([
        (cwd, cwd.parent / repo_name),
        (cwd / "Day3", cwd / repo_name),
        (cwd.parent / "Day3", cwd),
    ])

for day3_dir, repo_dir in path_pairs:
    backend_dir = repo_dir / "smartlearn-backend"
    if (
        (day3_dir / "pdf1.pdf").exists()
        and (day3_dir / "pdf2.pdf").exists()
        and (backend_dir / "services" / "rag.py").exists()
    ):
        DAY3_DIR = day3_dir.resolve()
        REPO_DIR = repo_dir.resolve()
        BACKEND_DIR = backend_dir.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not locate Day3/pdf1.pdf and smartLearn-AI/smartlearn-backend/services/rag.py from the current working directory."
    )

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from services import rag as rag_module

rag = importlib.reload(rag_module)
ARTIFACT_ROOT = DAY3_DIR / "artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

ACTIVE_CHUNK_MODE = "character_overlap"
CHUNK_SIZE = 700
OVERLAP = 120
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3
CANDIDATE_POOL = 60

print("Day3 folder:", DAY3_DIR)
print("Repository folder:", REPO_DIR)
print("Backend folder:", BACKEND_DIR)
print("Artifact root:", ARTIFACT_ROOT)


Day3 folder: C:\Users\nanfe\aiworkshop\Day3
Repository folder: C:\Users\nanfe\aiworkshop\smartLearn-AI
Backend folder: C:\Users\nanfe\aiworkshop\smartLearn-AI\smartlearn-backend
Artifact root: C:\Users\nanfe\aiworkshop\Day3\artifacts


> **Expected output:** the notebook prints the Day 3 folder, the artifact root, and the current embedding device.


## 2.3 Vibe Code the FAISS Helpers

### Purpose

This step adds the local vector index layer that sits between embeddings and answer generation.

### What Is FAISS?

<img src="https://www.designveloper.com/wp-content/uploads/2025/09/what-is-faiss-1-1024x614.webp" alt="img" style="zoom:50%;" />

FAISS is a library for **similarity search over dense vectors** by Meta AI. In a RAG pipeline, each chunk embedding becomes one vector, and FAISS stores those vectors in an index so we can quickly look up the chunks most similar to a question embedding.

#### Why It Matters

- **Fast retrieval:** one index can answer many questions without re-scanning every chunk
- **Clear trade-offs:** simple indexes can be exact, while larger indexes can trade a little accuracy for more speed or lower memory use
- **RAG-friendly output:** search returns top-k matches that we can map back to chunk text and page numbers

#### Common Design

At a high level, FAISS separates **storage** from **search**. We first build an index from embedding vectors, then reuse that index for retrieval. A flat index compares against all stored vectors and is exact. More advanced index families search only part of the database or compress vectors, which is useful when the collection becomes much larger.

#### Retrieval Flow

1. Turn document chunks into embeddings.
2. Build or load a FAISS index.
3. Add vectors to the index once.
4. Turn the user question into one query embedding.
5. Search top-k nearest vectors.
6. Map the returned ids back to chunk text and page numbers.

#### More about FAISS

https://faiss.ai/index.html


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: add index creation and reuse

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add or revise the FAISS helpers:
- `build_faiss_index(embeddings: np.ndarray) -> faiss.Index`: input normalized embedding vectors; output a searchable FAISS index.
- `save_faiss_index(index, index_path: str | Path) -> None`: input one FAISS index and output path; write the binary `.faiss` file to disk.
- `load_faiss_index(index_path: str | Path) -> faiss.Index`: input one saved `.faiss` path; output the FAISS index loaded back into memory.
- `ensure_index(document_id: str, pdf_name: str, pages: list[dict] | None = None, pdf_path: str | Path | None = None, chunk_mode: str = "character_overlap", model_name: str = "sentence-transformers/all-MiniLM-L6-v2", chunk_size: int = 700, overlap: int = 120, batch_size: int = 32, artifact_root: str | Path | None = None) -> dict`: input one PDF/page source and active settings; output a bundle with chunks, embeddings, manifest, FAISS index, and saved paths.

Also keep these reused Lab A helpers available because the next check cell calls them directly:
- `get_device() -> str`: input nothing; output `"cpu"` or `"cuda"` for setup display.
- `load_json(input_path: str | Path) -> Any`: input one JSON artifact path; output the loaded Python object.
- `extract_pages_for_rag(file_path: str | Path, page_limit: int | None = None) -> list[dict]`: input one PDF path; output page records like `[{"page": 1, "text": "..."}]`.
- `relative_path_str(path: str | Path, base: str | Path) -> str`: input one artifact path and base folder; output a shorter display path.
- `prepare_rag_document(document_id: str, filename: str, pages: list[dict], chunk_mode: str = "character_overlap", chunk_size: int = 700, overlap: int = 120, model_name: str = "sentence-transformers/all-MiniLM-L6-v2", batch_size: int = 32, artifact_root: str | Path | None = None) -> dict`: input one document id, filename, and page records; output a server-style document record with pages, chunks, empty history, and RAG index metadata.

Requirements:
- use normalized embeddings with a local FAISS inner-product index
- save a .faiss file plus a small metadata file
- rebuild the index only when the signature changes
- keep artifact_root configurable so the notebook can stay inside Day3/artifacts
- do not edit any other project file

Before editing, explain why normalized embeddings + inner product are enough for this notebook.
```

### Review map after the edit

Find these ideas in the diff:
- where the FAISS index dimension comes from;
- where the binary index file is written;
- where the index signature is checked before reuse.

**Completion evidence:** one prepared document record points to a real `.faiss` file on disk.


In [4]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

pdf1_pages_path = ARTIFACT_ROOT / "raw_pages" / "pdf1_pages.json"
if pdf1_pages_path.exists():
    pdf1_pages = rag.load_json(pdf1_pages_path)
    print("Loaded cached pages from:", rag.relative_path_str(pdf1_pages_path, DAY3_DIR))
else:
    pdf1_pages = rag.extract_pages_for_rag(DAY3_DIR / "pdf1.pdf")
    print("Rebuilt pages from pdf1.pdf")

document_pdf1 = rag.prepare_rag_document(
    document_id="pdf1",
    filename="pdf1.pdf",
    pages=pdf1_pages,
    chunk_mode=ACTIVE_CHUNK_MODE,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    model_name=EMBED_MODEL_NAME,
    batch_size=32,
    artifact_root=ARTIFACT_ROOT,
)

{
    "index_path": rag.relative_path_str(document_pdf1['artifacts']["index"], DAY3_DIR),
    "chunk_path": rag.relative_path_str(document_pdf1['artifacts']["chunks"], DAY3_DIR),
    "num_chunks": document_pdf1["chunk_size"],
    "embedding_dim": document_pdf1["embedding_dim"],
}


Rebuilt pages from pdf1.pdf


{'index_path': 'artifacts\\pdf1__index__character_overlap__s700__o120__sentence-transformers-all-MiniLM-L6-v2.faiss',
 'chunk_path': 'artifacts\\pdf1__chunks__character_overlap__s700__o120.json',
 'num_chunks': 700,
 'embedding_dim': 384}

> **Checkpoint 2 - The index exists:** `document_pdf1["artifacts"]["index"]` points to a saved `.faiss` file, and the document record already contains chunk metadata for later retrieval.


## 2.4 Vibe Code Top-k Retrieval and Local Answer Extraction

### Purpose

Now use the index to retrieve evidence and return a short local answer from the best retrieved sentence. Retrieval usually happens before answer generation: first find the most similar chunks, then read only those chunks for evidence. This keeps the answer tied to document text instead of the whole PDF.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: retrieve first, answer second

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add or revise these retrieval helpers:
- `keyword_set(text: str) -> set[str]`: input one question or chunk text; output lightweight lexical tokens for simple reranking.
- `search_bundle(question: str, bundle: dict, top_k: int = 3, candidate_pool: int = 60, batch_size: int = 1, history: list[dict] | None = None) -> list[dict]`: input one question and an in-memory index bundle; output top-k hits with page, chunk id, text, and score fields.
- `search_document(question: str, document: dict, top_k: int = 3, candidate_pool: int = 60, history: list[dict] | None = None) -> list[dict]`: input one question and a prepared document record; load the saved FAISS index, run retrieval, and output top-k hits with page and score information.
- `split_sentences(text: str) -> list[str]`: input retrieved chunk text; output candidate answer sentences.
- `best_sentence_answer(question: str, hits: list[dict]) -> str`: input one question and retrieved hits; output one short local answer sentence with a page tag when possible.

Requirements:
- embed the question with the same embedding model
- retrieve top-k chunks from the saved FAISS index
- keep page, chunk_id, text, and scores in each hit
- a small lexical rerank is allowed if it stays easy to explain
- best_sentence_answer should return one short answer sentence with a page tag when possible
- do not edit any other project file

Before editing, explain what information a single retrieval hit should carry.
```

### Review map after the edit

Locate:
- where the query embedding is created;
- where the retrieved index positions map back to chunk records;
- where page numbers stay attached to the final hits.

**Completion evidence:** asking one question about `pdf1.pdf` returns top-k chunks plus a short local answer.


In [6]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

question_pdf1 = "Which single-hop datasets are used in the paper?"
hits_pdf1 = rag.search_document(
    question_pdf1,
    document_pdf1,
    top_k=TOP_K,
    candidate_pool=CANDIDATE_POOL,
)

pd.set_option("display.max_colwidth", 200)
hits_pdf1_df = pd.DataFrame(hits_pdf1)
print(hits_pdf1_df)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4287.46it/s]


   page  chunk_id  \
0     6        49   
1    28       190   
2     8        66   

                                                                                                                                                                                                      text  \
0  le benchmark datasets. For single-hop and multi-hop QA, we mainly fol- low HippoRAG 2 (Jim´enez Guti´errez et al., 2025) and use 1,000 randomly sampled queries from (1) single-hop: NQ (Kwiatkowski...   
1  .81GB 656.7MB(102.6×)37.89MB(1,778×) 1.34GB34.10MB(40.24×) Tree indexing time (s)15,996 13,957(1.15×)848(18.86×) ∼280,0002,261(∼124×) Abstraction time (s)∼205,000∼200,000∼89,400 ∼1,800,000∼1,400,0...   
2   reorganize and enrich their generated queries at each retrieval attempt. This leads to performance degradation across multi-hop datasets, with a 2% F1 drop on MuSiQue. Therefore, the thematic key...   

    score  vector_score  
0  0.5236        0.5123  
1  0.4886        0.5013  
2  0.4658   

In [7]:
answer_pdf1 = rag.best_sentence_answer(question_pdf1, hits_pdf1)
print("Question:", question_pdf1)
print("Local answer:", answer_pdf1)
print("Pages:", sorted({hit["page"] for hit in hits_pdf1}))

Question: Which single-hop datasets are used in the paper?
Local answer: le benchmark datasets. (page 6)
Pages: [6, 8, 28]


> **Checkpoint 3 - Local retrieval works:** you can point to the top retrieved pages and show one short local answer sentence grounded in those hits.


## 2.5 Vibe Code the Project-Facing Wrapper

### Purpose

Lab B should already move toward the Day 2 app shape. The backend later needs one document record and one answer function, not notebook-only glue code.


The app does not want notebook steps one by one; it wants one clean function call. This wrapper bundles chunk loading, index reuse, retrieval, and answer formatting into one project-facing path.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: package retrieval for the app

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add or revise these project-facing helpers:
- `prepare_rag_document(document_id: str, filename: str, pages: list[dict], chunk_mode: str = "character_overlap", chunk_size: int = 700, overlap: int = 120, model_name: str = "sentence-transformers/all-MiniLM-L6-v2", batch_size: int = 32, artifact_root: str | Path | None = None) -> dict`: input uploaded page records and active settings; output one server-side document record with pages, chunks, index paths, and empty history.
- `extract_citations(answer: str, hits: list[dict] | None = None) -> list[int]`: input an answer and optional retrieved hits; output numeric PDF page citations.
- `build_sources(hits: list[dict]) -> list[dict]`: input retrieval hits; output frontend-friendly source objects with page, chunk id, score, and preview text.
- `answer_document(document: dict, question: str, top_k: int = 3, candidate_pool: int = 60, answer_model: str = "openrouter/free") -> dict`: input a prepared document and one question; output `answer`, `citations`, and `sources` after retrieval and optional LLM answering.
- `append_history(document: dict, question: str, result: dict) -> list[dict]`: input the stored document, user question, and answer result; output the updated in-memory history list.

Requirements:
- prepare_rag_document should return a dict that can later be stored in documents[document_id]
- answer_document should return at least answer, citations, and sources
- if OPENROUTER_API_KEY is missing, answer_document should still return a local extracted answer
- if the API key exists, it may use the retrieved chunks to ask the LLM
- sources should keep page numbers so the frontend can later make clickable links
- do not edit main.py, pdf.py, llm.py, or frontend files in this step

Before editing, describe the minimum shape of one stored document record.
```

### Review map after the edit

Check that:
- the stored record contains both chunk metadata and index path information;
- page numbers survive into `sources`;
- the answer path still works without an API key.

**Completion evidence:** one `answer_document(...)` call returns an answer, a numeric citation list, and a source list.


In [8]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

result_pdf1 = rag.answer_document(
    document_pdf1,
    question_pdf1,
    top_k=TOP_K,
    candidate_pool=CANDIDATE_POOL,
)

print("Question:", question_pdf1)
print("Answer:", result_pdf1["answer"])
print("Citations:", result_pdf1["citations"])
pd.DataFrame(result_pdf1["sources"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6592.80it/s]


Question: Which single-hop datasets are used in the paper?
Answer: Based on the provided context, the paper uses two single-hop datasets: **NQ** (Natural Questions, Kwiatkowski et al., 2019) and **PopQA** (Mallen et al., 2023) [Page 6].
Citations: [6]


,page,chunk_id,score,preview
0,6,49,0.5236,"le benchmark datasets. For single-hop and multi-hop QA, we mainly fol- low HippoRAG 2 (Jim´enez Guti´errez et al., 2025) and use 1,000 randomly sampled queries from (1) single-hop: NQ (Kwiatkowski..."
1,28,190,0.4886,".81GB 656.7MB(102.6×)37.89MB(1,778×) 1.34GB34.10MB(40.24×) Tree indexing time (s)15,996 13,957(1.15×)848(18.86×) ∼280,0002,261(∼124×) Abstraction time (s)∼205,000∼200,000∼89,400 ∼1,800,000∼1,400,0..."
2,8,66,0.4658,"reorganize and enrich their generated queries at each retrieval attempt. This leads to performance degradation across multi-hop datasets, with a 2% F1 drop on MuSiQue. Therefore, the thematic key..."


## 2.6 Prepare the 100+ Page Example

### Purpose

Now run the same pipeline on the larger PDF. This is the main long-document milestone for Day 3.


In [9]:
pdf2_pages = rag.extract_pages_for_rag(DAY3_DIR / "pdf2.pdf")
document_pdf2 = rag.prepare_rag_document(
    document_id="pdf2",
    filename="pdf2.pdf",
    pages=pdf2_pages,
    chunk_mode=ACTIVE_CHUNK_MODE,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    model_name=EMBED_MODEL_NAME,
    batch_size=32,
    artifact_root=ARTIFACT_ROOT,
)

{
    "num_pages": len(document_pdf2["pages"]),
    "num_chunks": document_pdf2["chunk_size"],
    "index_path": rag.relative_path_str(document_pdf2['artifacts']["index"], DAY3_DIR),
}

{'num_pages': 202,
 'num_chunks': 700,
 'index_path': 'artifacts\\pdf2__index__character_overlap__s700__o120__sentence-transformers-all-MiniLM-L6-v2.faiss'}

> **Checkpoint 4 - The large document is indexed:** `pdf2.pdf` produces a stored document record with a reusable FAISS index path.


## 2.7 Ask a Large-PDF Question

### Purpose

Use one question that should be answerable only after retrieval from the long guide.


In [10]:
question_pdf2 = "Which section covers SFT, DPO, and GRPO?"
hits_pdf2 = rag.search_document(
    question_pdf2,
    document_pdf2,
    top_k=TOP_K,
    candidate_pool=CANDIDATE_POOL,
)

hits_pdf2_df = pd.DataFrame(hits_pdf2)
print(hits_pdf2_df)


   page  chunk_id  \
0   137       508   
1   112       438   
2   131       495   

                                                                                                                                                                                                      text  \
0  g simple to implement, stable in practice, and effective even with modest amounts of preference data. As a result, DPO has become the default method to improve SFT models before reaching for more ...   
1  sually get meaningful gains without needing to burn a bonfire of silicon, and in fraction of the time required for RL. It’s stable: Unlike RL, which is notoriously sensitive to reward design and h...   
2  ng happens before SFT on the base model, but whether it will be beneficial often only becomes clear after you’ve run initial SFT experiments and identified performance gaps. In practice, you’ll of...   

    score  vector_score  
0  0.4849        0.4911  
1  0.4144        0.3969  
2  0.4132   

In [11]:
result_pdf2 = rag.answer_document(
    document_pdf2,
    question_pdf2,
    top_k=TOP_K,
    candidate_pool=CANDIDATE_POOL,
)

print("Question:", question_pdf2)
print("Answer:", result_pdf2["answer"])
print("Citations:", result_pdf2["citations"])
print("Pages:", sorted({hit["page"] for hit in hits_pdf2}))

Question: Which section covers SFT, DPO, and GRPO?
Answer: The provided context mentions SFT and DPO, but it does not include any section that discusses GRPO. Therefore, the document does not provide enough information about a section covering GRPO.
Citations: []
Pages: [112, 131, 137]


> **Checkpoint 5 - Long-PDF retrieval works:** the returned pages and answer come from retrieved chunks, not from a full-document prompt.


## 2.8 Use a Small Short-Answer Evaluation Set

### Purpose

Keep scoring simple. Each question should have a short answer string that must come from the PDFs themselves.


<!-- BEGINNER-WALKTHROUGH -->

### Beginner walkthrough: judge the question set before running it

For each evaluation question, check three rules:
- the answer is short enough to score by string match;
- the answer appears in the PDF text;
- the question is specific enough that retrieval matters.

If a question could be answered from general world knowledge, replace it.

**Completion evidence:** every question in the set has at least one exact or near-exact gold answer string.


In [15]:
EVAL_SET = [
    {
        "pdf_name": "pdf1.pdf",
        "question": "What is the name of the sparse retriever used in this paper?",
        "answers": ["BM25"],
        "answer_pages":[2,4,5,6]
    },
    {
        "pdf_name": "pdf1.pdf",
        "question": "Which model is used as the R&A agent by default?",
        "answers": ["Llama-3-70B", "L3-70B"],
        "answer_pages":[6,8,9]
    },
    {
        "pdf_name": "pdf1.pdf",
        "question": "What is the metric used for the document-level summarization task?",
        "answers": ["ROUGE-L", "Recall-Oriented Understudy for Gisting Evaluation using the Longest Common Subsequence"],
        "answer_pages":[6,8]
    },
    {
        "pdf_name": "pdf2.pdf",
        "question": "Which model name appears before the phrase trained on 11T tokens?",
        "answers": ["SmolLM3"],
        "answer_pages":[2,9,20,]
    },
    {
        "pdf_name": "pdf2.pdf",
        "question": "Which section heading covers SFT, DPO, and GRPO?",
        "answers": ["Post-training"],
        "answer_pages":[3,8,18,103,104,105,106]
    },
    {
        "pdf_name": "pdf2.pdf",
        "question": "Which heading is shown as Training Compass: Why -> What -> How?",
        "answers": ["Training Compass"],
        "answer_pages":[3,4,21,103,104,105,200]
    },
]


## 2.9 Vibe Code the Evaluation Helper

### Purpose

The evaluation helper should answer one question repeatedly and summarize hit / miss results. A small evaluation loop is a quick way to see whether retrieval is actually returning the right evidence. Here we use short-answer questions so the result is easy to check by eye and by Exact Match (EM).


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: build the smallest useful evaluation helper

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add or revise the helpers for a simple retrieval evaluation:
- `normalize_for_match(text: str) -> str`: input extracted text or one gold answer; output normalized text for simple string-based scoring.
- `contains_any_answer(text: str, answers: list[str]) -> bool`: input one text block and acceptable answers; output whether any answer appears after normalization.
- `evaluate_questions(eval_set: list[dict], documents_by_name: dict[str, dict], top_k: int = 3, candidate_pool: int = 60) -> pandas.DataFrame`: input the evaluation records and prepared documents; output one table row per question with pages, local answer, `retrieval_hit`, and `answer_hit`.

Requirements:
- evaluate_questions should accept a list of question records and a mapping from pdf_name to prepared document records
- each row should record the local answer, retrieved pages, retrieval_hit, and answer_hit
- keep the logic easy to explain to a beginner
- do not edit other project files

Before editing, explain the difference between retrieval_hit and answer_hit in one sentence each.
```

**Completion evidence:** the evaluation result is a table, not only a print statement.


In [16]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

documents_by_name = {
    "pdf1.pdf": document_pdf1,
    "pdf2.pdf": document_pdf2,
}

eval_df = rag.evaluate_questions(
    EVAL_SET,
    documents_by_name,
    top_k=TOP_K,
    candidate_pool=CANDIDATE_POOL,
)
eval_df

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7094.98it/s]


,question,pdf_name,gold_pages,retrieved_pages,local_answer,retrieval_hit,answer_hit
0,What is the name of the sparse retriever used in this paper?,pdf1.pdf,"[2, 4, 5, 6]","[4, 6, 26]","The sparse retriever used in this paper is BM25 (Robertson et al., 2009). This is mentioned in both the context on page 6, which states ""the sparse retriever BM25 (Robertson et al., 2009)"", and on...",True,True
1,Which model is used as the R&A agent by default?,pdf1.pdf,"[6, 8, 9]","[8, 9, 26]","Based on the provided text, the document does not provide enough information to determine which model is used as the R&A agent by default. The text mentions several models used as R&A agents in di...",True,False
2,What is the metric used for the document-level summarization task?,pdf1.pdf,"[6, 8]","[6, 22, 23]",The metric used for the document-level summarization task is ROUGE-L. [Page 6],True,True
3,Which model name appears before the phrase trained on 11T tokens?,pdf2.pdf,"[2, 9, 20]","[9, 64, 85]","The document does not contain the phrase “trained on 11T tokens,” so there is no model name that appears before it in the provided context.",True,False
4,"Which section heading covers SFT, DPO, and GRPO?",pdf2.pdf,"[3, 8, 18, 103, 104, 105, 106]","[112, 131, 137]",User Safety: safe,False,False
5,Which heading is shown as Training Compass: Why -> What -> How?,pdf2.pdf,"[3, 4, 21, 103, 104, 105, 200]","[3, 105]",The document does not provide enough information to identify a heading titled “Training Compass: Why → What → How.”,True,True


In [17]:
retrieval_acc = float(eval_df["retrieval_hit"].mean())
answer_acc = float(eval_df["answer_hit"].mean())

print(f"Retrieval hit rate: {retrieval_acc:.2%}")
print(f"Local answer hit rate: {answer_acc:.2%}")

report_path = ARTIFACT_ROOT / "reports" / "section2_eval_results.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(report_path, index=False, encoding="utf-8-sig")
print("Saved report:", rag.relative_path_str(report_path, DAY3_DIR))

Retrieval hit rate: 83.33%
Local answer hit rate: 50.00%
Saved report: artifacts\reports\section2_eval_results.csv


## 2.10 Commit the Local Retrieval Milestone

### Purpose

This section added the first local vector retrieval path, a local answer extraction path, and a small evaluation report. Save this working state before page mapping and multi-turn chat add more moving parts.

### Step 1: Confirm the section evidence

Before Git commands, confirm:

- `index.faiss` exists under `Day3/artifacts/pdf1/character_overlap_c700_o120_sentence-transformers_all-MiniLM-L6-v2`;
- `index.faiss` exists under `Day3/artifacts/pdf2/character_overlap_c700_o120_sentence-transformers_all-MiniLM-L6-v2`;
- `section2_eval_results.csv` exists under `Day3/artifacts/reports/`;
- one known question returns sensible page numbers;
- `smartlearn-backend/services/rag.py` now contains retrieval and evaluation helpers.

If one item is missing, fix that first.

### Step 2: Inspect the working tree

From the `smartLearn-AI` repository root, run:

```bash
git status
```

These files should not be staged:

- `Day3/artifacts/` ? generated indexes and reports;
- `.venv/` ? local environment;
- `__pycache__/` and `*.pyc` ? generated cache;
- uploaded PDFs or copied workshop documents;
- `.env` ? local secrets.

### Step 3: Review and stage only the retrieval code

Run:

```bash
git diff -- smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git add smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git status
git diff --staged
```

Confirm the staged diff shows:

- FAISS index helpers;
- search and local answer helpers;
- the evaluation loop and only the dependency updates you can explain.

### Step 4: Create and verify the commit

Run:

```bash
git commit -m "feat: add local FAISS retrieval pipeline"
git log -1 --oneline
git status
```

> **Expected output:** One commit records the local retrieval milestone, while generated FAISS files and CSV reports stay untracked.


In [ ]:
git status
git diff -- smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git add smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git status
git diff --staged
git commit -m "feat: add local FAISS retrieval pipeline"
git log -1 --oneline
git status


> ✅ **Checkpoint 10 — The local retrieval milestone is committed:** The repo records the FAISS-based retrieval path and evaluation helpers, while generated indexes and reports remain outside the commit.
>
> **If it does not match:** Unstage generated files, rerun one retrieval example, and commit only after you can explain the staged search and evaluation logic.


## Lab B Checkpoint

- [ ] I can explain what FAISS is doing in this project.
- [ ] I prepared a stored document record for `pdf1.pdf`.
- [ ] I prepared a stored document record for `pdf2.pdf`.
- [ ] I can retrieve top-k chunks and read their page numbers.
- [ ] I can get a local answer even without an API key.
- [ ] I saved `section2_eval_results.csv`.

**Expected output**
- `artifacts/pdf1/character_overlap_c700_o120_sentence-transformers_all-MiniLM-L6-v2/index.faiss`
- `artifacts/pdf2/character_overlap_c700_o120_sentence-transformers_all-MiniLM-L6-v2/index.faiss`
- `artifacts/reports/section2_eval_results.csv`

**Homework for Day 4**
1. Add one new evaluation question for each PDF.
2. Change `TOP_K` from `3` to `5` and compare the hit rate.
3. Write one sentence about when a retrieval hit can still lead to a weak answer.


## Appendix A: Try a Local Chroma Collection

This appendix adds a collection-based storage option as an optional extension. Keep the main Lab B FAISS path unchanged and do this only after the main path works.

### What Is Chroma?

<img src="https://www.trychroma.com/_next/static/media/chroma-wordmark.0~1c352v-zy35.svg?dpl=dpl_PrCRRwGVEvjGidsMuQXwkfPQqGkw" alt="img" style="zoom:200%;" />

Chroma is an open-source embedding database. Instead of storing only vectors in one local FAISS index file, Chroma organizes retrieval data in collections and can keep embeddings, documents, ids, and metadata together.

#### Why It Matters

- one collection can keep chunk text and page metadata together
- query results can return documents and metadata in one place
- metadata filtering is built in, which is useful for later application-style retrieval

#### Common Design

In this appendix, one collection can represent one document or one experiment. We add chunk ids, chunk text, page numbers, and embeddings into the collection, then query that collection for the top-k most similar chunks.

#### Retrieval Flow

1. Reuse the chunk and embedding files from the main Lab B path.
2. Create or reopen one Chroma collection.
3. Add chunk ids, documents, metadata, and embeddings.
4. Query the collection for top-k matches.
5. Map the returned rows back to the same answer shape used in the main path.

#### More about Chroma

https://docs.trychroma.com/

Install first if needed:

```bash
pip install chromadb
```

Add it to `smartLearn-AI/smartlearn-backend/requirements.txt`.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: add an optional Chroma branch without replacing FAISS

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add a small optional Chroma branch that reuses the same chunk and embedding data from Lab B.

Use these helper entry names:
- `_require_chromadb() -> module`: input nothing; output the imported `chromadb` module or raise a clear ImportError if it is missing.
- `build_chroma_collection(document_id: str, chunks: list[dict], embeddings: np.ndarray, persist_dir: str | Path) -> dict`: input one document id, chunk records, embedding matrix, and storage folder; output collection metadata such as collection name and item count.
- `query_chroma_collection(document_id: str, query_embedding: np.ndarray, persist_dir: str | Path, top_k: int) -> list[dict]`: input one query vector and collection folder; output top-k matches with `chunk_id`, `page`, `text`, and score fields.
- `search_document_with_chroma(question: str, document: dict, persist_dir: str | Path, top_k: int = 3, batch_size: int = 1) -> list[dict]`: input one question and prepared document; output top-k Chroma hits in the same general shape as FAISS hits.
- `answer_document_with_chroma(document: dict, question: str, persist_dir: str | Path, top_k: int = 3, answer_model: str = "openrouter/free") -> dict`: input one prepared document and question; output the same `answer`, `citations`, and `sources` shape as the FAISS path.

Reuse these existing helper names in the appendix check cells:
- `ensure_artifact_dirs(artifact_root: str | Path | None = None) -> dict[str, Path]`: input an optional artifact root; output all artifact folders, including the Chroma storage folder path.
- `embed_texts(texts: list[str], model_name: str, model_cache_dir: str | Path | None = None, batch_size: int = 32) -> np.ndarray`: input appendix question text and model settings; output normalized query embedding vectors.

Implementation requirements:
- add `chromadb` to `smartLearn-AI/smartlearn-backend/requirements.txt`
- keep `build_faiss_index`, `save_faiss_index`, `load_faiss_index`, `ensure_index`, `search_bundle`, and `answer_document` unchanged as the default Lab B path
- treat Chroma as an appendix-only branch, not as a replacement for the FAISS branch
- store page number and any other needed fields as metadata in the collection
- keep the caller-facing retrieval result recognizable: top-k chunks, page numbers, and a short local answer path
- if you add any backend selector parameter, default it to `"faiss"` so the main notebook path keeps working

After editing, explain:
- where the Chroma collection is stored
- which function builds it
- which function queries it
- what stays unchanged for the caller
```

A good optional branch keeps these outputs recognizable:
- top-k retrieved chunks
- page numbers
- short local answer path

**Completion evidence:** you can point to one optional Chroma code path in `rag.py` and explain why the main FAISS notebook path still runs unchanged.


In [ ]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

import numpy as np

CHROMA_PERSIST_DIR = rag.ensure_artifact_dirs(ARTIFACT_ROOT)["chroma"]

chroma_build = rag.build_chroma_collection(
    document_id=document_pdf1["document_id"],
    chunks=document_pdf1["chunks"],
    embeddings=np.load(ARTIFACT_ROOT / document_pdf1["artifacts"]["embeddings"], allow_pickle=False),
    persist_dir=CHROMA_PERSIST_DIR,
)

{
    "persist_dir": rag.relative_path_str(CHROMA_PERSIST_DIR, DAY3_DIR),
    "collection_name": chroma_build["collection_name"],
    "item_count": chroma_build["item_count"],
}

In [ ]:
appendix_question = next(
    item["question"] for item in EVAL_SET if item["pdf_name"] == "pdf1.pdf"
)

appendix_query_vector = rag.embed_texts(
    [appendix_question],
    model_name=document_pdf1["model_name"],
    model_cache_dir=document_pdf1["model_source"],
    batch_size=1,
)

chroma_hits = rag.query_chroma_collection(
    document_id=document_pdf1["document_id"],
    query_embedding=appendix_query_vector,
    persist_dir=CHROMA_PERSIST_DIR,
    top_k=TOP_K,
)

chroma_hits_df = pd.DataFrame(chroma_hits)
chroma_hits_df

In [ ]:
from services import rag as rag_module
rag = importlib.reload(rag_module)

chroma_result = rag.answer_document_with_chroma(
    document_pdf1,
    appendix_question,
    persist_dir=CHROMA_PERSIST_DIR,
    top_k=TOP_K,
)

print(chroma_result["answer"])
print("Citations:", chroma_result["citations"])

pd.DataFrame(chroma_result["sources"])

### Appendix A Checkpoint

- [ ] I can build one persistent Chroma collection from the same chunks and embeddings.
- [ ] I can query top-k chunk hits and still read page numbers.
- [ ] I can get the same answer / citations / sources shape from an optional Chroma branch.
- [ ] I know the FAISS path is still the default Lab B path.


### Bridge to Lab C

The important handoff for the next lab is already visible here:
- `answer_document(...)` returns `citations`
- `answer_document(...)` returns `sources` with page numbers
- one stored document record can also keep `history`

Lab C will attach those fields to the backend route and the frontend page links.
